# 5회차: MLOps - ML과 DevOps의 만남

**DevOps 엔지니어를 위한 ML 교육 (5/5)**

---

## 학습 목표

- ML 대회와 실무의 차이를 이해한다
- MLOps의 전체 파이프라인을 파악한다
- DevOps 경험을 MLOps에 어떻게 활용할 수 있는지 알아본다
- 이번 교육에서 배운 내용을 실무에 연결한다

---

# Part 1: 이론 (40분)

## 1. 대회 vs 실무

지난 4회차 동안 우리는 Kaggle 대회 형식으로 ML을 배웠습니다.  
하지만 실무에서의 ML은 대회와 **근본적으로 다릅니다.**

### 일회성 예측 vs 지속적 운영

| 구분 | Kaggle 대회 | 실무 ML 시스템 |
|------|------------|---------------|
| **목표** | 리더보드 점수 최적화 | 비즈니스 가치 창출 |
| **데이터** | 고정된 데이터셋 | 지속적으로 변하는 데이터 |
| **학습** | 한 번 학습, 한 번 예측 | 주기적 재학습 필요 |
| **배포** | CSV 파일 제출 | API 서빙, 실시간 추론 |
| **모니터링** | 없음 | 24/7 모니터링 필수 |
| **코드 품질** | 노트북에서 실행되면 OK | 테스트, CI/CD, 코드 리뷰 |
| **성공 기준** | 정확도 숫자 하나 | 레이턴시, 처리량, 안정성 등 복합 지표 |

### DevOps 비유로 이해하기

대회에서 실무로의 전환은 DevOps에서 이런 것과 비슷합니다:

- **대회** = 로컬에서 스크립트 한 번 실행하는 것
- **실무** = 프로덕션 서비스를 24/7 운영하는 것

로컬에서 잘 돌아가는 스크립트와 프로덕션 서비스 사이의 간극,  
그 간극을 메우는 것이 바로 여러분이 잘 아는 **DevOps**이고,  
ML 영역에서 그 역할을 하는 것이 **MLOps**입니다.

## 2. "ML 코드는 전체 시스템의 5%"

Google의 유명한 논문 *"Hidden Technical Debt in Machine Learning Systems"* (2015)에서  
실제 ML 시스템에서 ML 코드가 차지하는 비중이 얼마나 작은지를 보여주었습니다.

### ML 시스템의 실체

```
┌─────────────────────────────────────────────────────────────────────┐
│                        ML 시스템 전체 구성                           │
│                                                                     │
│  ┌─────────────┐  ┌──────────────┐  ┌─────────────────────────┐   │
│  │  데이터 수집   │  │  데이터 검증   │  │    피처 추출 / 변환       │   │
│  └─────────────┘  └──────────────┘  └─────────────────────────┘   │
│                                                                     │
│  ┌──────────────────┐  ┌──────────────┐  ┌────────────────────┐   │
│  │ 프로세스 관리 도구  │  │  분석 도구    │  │  리소스 관리         │   │
│  └──────────────────┘  └──────────────┘  └────────────────────┘   │
│                                                                     │
│                    ┌───────────────┐                                │
│                    │               │                                │
│                    │   ML 코드      │  <-- 전체의 약 5%              │
│                    │               │                                │
│                    └───────────────┘                                │
│                                                                     │
│  ┌──────────────┐  ┌──────────────┐  ┌─────────────────────────┐   │
│  │  서빙 인프라   │  │  모니터링      │  │    설정 (Configuration)  │   │
│  └──────────────┘  └──────────────┘  └─────────────────────────┘   │
│                                                                     │
│  ┌────────────────────────────────────────────────────────────┐    │
│  │              머신 리소스 관리 / 인프라 계층                     │    │
│  └────────────────────────────────────────────────────────────┘    │
└─────────────────────────────────────────────────────────────────────┘
```

### 나머지 95%는 무엇인가?

| 영역 | 설명 | DevOps 대응 |
|------|------|------------|
| **데이터 수집** | 다양한 소스에서 데이터 수집, 정제 | ETL 파이프라인, Kafka, Airflow |
| **데이터 검증** | 데이터 품질 검사, 스키마 검증 | 입력 데이터 validation |
| **피처 추출** | 원본 데이터를 모델 입력으로 변환 | 데이터 전처리 파이프라인 |
| **설정 관리** | 하이퍼파라미터, 모델 설정 | ConfigMap, 환경변수 관리 |
| **서빙 인프라** | 모델을 API로 배포, 로드 밸런싱 | Kubernetes, Docker, Nginx |
| **모니터링** | 모델 성능, 데이터 드리프트 감시 | Prometheus, Grafana, 알림 |
| **리소스 관리** | GPU/CPU 할당, 스케일링 | K8s 리소스 관리, 오토스케일링 |

> **핵심 메시지**: ML 엔지니어가 만드는 모델 코드보다,  
> 그것을 **안정적으로 운영하는 인프라**가 훨씬 더 크고 중요합니다.  
> 이것이 바로 DevOps 엔지니어가 MLOps에서 핵심적인 이유입니다.

## 3. MLOps 파이프라인

MLOps는 크게 4개의 파이프라인으로 구성됩니다.

### 3.1 데이터 파이프라인

```
┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│ 데이터    │───>│ 추출     │───>│ 변환     │───>│ 적재     │
│ 소스들    │    │ Extract  │    │Transform │    │  Load    │
└──────────┘    └──────────┘    └──────────┘    └──────────┘
     │                                               │
     │          ┌──────────────────────┐              │
     └─────────>│  데이터 품질 관리      │<─────────────┘
                │  - 스키마 검증         │
                │  - 이상치 탐지         │
                │  - 데이터 드리프트 감지  │
                └──────────────────────┘
```

**DevOps 비유**: 로그 수집 파이프라인 (Fluentd/Logstash → 변환 → Elasticsearch)과 동일한 패턴입니다.

**주요 도구**: Apache Airflow, dbt, Great Expectations, Apache Kafka

---

### 3.2 학습 파이프라인

```
┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────────┐
│ 데이터    │───>│ 피처     │───>│ 모델     │───>│ 모델          │
│ 준비     │    │ 엔지니어링 │    │ 학습     │    │ 레지스트리     │
└──────────┘    └──────────┘    └──────────┘    └──────────────┘
                                     │
                               ┌─────┴─────┐
                               │ 실험 추적   │
                               │ - 파라미터  │
                               │ - 메트릭    │
                               │ - 아티팩트  │
                               └───────────┘
```

**DevOps 비유**: CI 파이프라인과 유사합니다.
- 데이터 준비 = 소스 코드 체크아웃
- 피처 엔지니어링 = 의존성 설치
- 모델 학습 = 빌드 + 테스트
- 모델 레지스트리 = 아티팩트 저장소 (Docker Registry, Artifactory)

**주요 도구**: MLflow, Weights & Biases, DVC, Neptune.ai

---

### 3.3 서빙 파이프라인

```
┌──────────────┐    ┌──────────┐    ┌──────────────┐
│ 모델          │───>│ 모델     │───>│ 프로덕션      │
│ 레지스트리     │    │ 검증     │    │ 서빙          │
└──────────────┘    └──────────┘    └──────┬───────┘
                                          │
                    ┌─────────────────────┐│
                    │ 배포 전략:            ││
                    │ - 카나리 배포         ││
                    │ - A/B 테스트         ││
                    │ - 블루/그린 배포      ││
                    │ - 섀도우 모드         ││
                    └─────────────────────┘│
                                          │
                    ┌─────────────────────┐│
                    │ 서빙 방식:            │┘
                    │ - 실시간 추론 (API)   │
                    │ - 배치 추론           │
                    │ - 스트리밍 추론       │
                    └─────────────────────┘
```

**DevOps 비유**: CD 파이프라인 + 배포 전략과 완전히 동일한 패턴입니다.  
카나리 배포, 블루/그린 배포 등 이미 알고 있는 개념을 그대로 적용할 수 있습니다.

**주요 도구**: TensorFlow Serving, NVIDIA Triton, Seldon Core, BentoML

---

### 3.4 모니터링

```
┌──────────────────────────────────────────────────────────────┐
│                      모니터링 대시보드                         │
├──────────────┬──────────────┬──────────────┬────────────────┤
│ 인프라 메트릭  │ 모델 메트릭   │ 데이터 메트릭  │ 비즈니스 메트릭 │
│              │              │              │                │
│ - GPU 사용률  │ - 정확도     │ - 입력 분포   │ - 전환율       │
│ - 레이턴시    │ - 예측 분포   │ - 피처 분포   │ - 매출 영향    │
│ - 처리량     │ - 에러율      │ - 결측값 비율  │ - 사용자 만족도 │
│ - 메모리     │ - 드리프트    │ - 데이터 양   │ - A/B 결과     │
└──────────────┴──────────────┴──────────────┴────────────────┘
```

**데이터 드리프트란?**

시간이 지남에 따라 입력 데이터의 분포가 학습 데이터와 달라지는 현상입니다.

- 예: 부동산 가격 예측 모델을 2019년 데이터로 학습했는데, 코로나 이후 시장 패턴이 완전히 바뀜
- DevOps 비유: 트래픽 패턴이 갑자기 바뀌어서 기존 오토스케일링 설정이 안 맞는 상황

**모델 성능 저하 탐지**

- 예측값의 분포 변화 모니터링
- 실제 정답(ground truth)이 들어오면 성능 재평가
- 임계값 기반 알림 설정

**주요 도구**: Prometheus + Grafana, Evidently AI, Whylogs, NannyML

## 4. DevOps 엔지니어의 MLOps 역할

DevOps 엔지니어는 MLOps에서 **가장 중요한 역할** 중 하나를 담당합니다.  
앞서 본 것처럼, ML 시스템의 95%는 인프라이기 때문입니다.

### 4.1 GPU 인프라 관리

```
┌─────────────────────────────────────────────────┐
│              Kubernetes 클러스터                   │
│                                                   │
│  ┌───────────┐  ┌───────────┐  ┌───────────┐    │
│  │ CPU 노드   │  │ CPU 노드   │  │ CPU 노드   │    │
│  │ (API 서빙) │  │ (전처리)   │  │ (모니터링) │    │
│  └───────────┘  └───────────┘  └───────────┘    │
│                                                   │
│  ┌───────────┐  ┌───────────┐                    │
│  │ GPU 노드   │  │ GPU 노드   │                    │
│  │ (학습)     │  │ (추론)     │                    │
│  │ A100 x 4  │  │ T4 x 2    │                    │
│  └───────────┘  └───────────┘                    │
│                                                   │
│  NVIDIA Device Plugin + GPU Operator              │
└─────────────────────────────────────────────────┘
```

**핵심 업무**:
- GPU 노드 프로비저닝 및 관리
- NVIDIA GPU Operator, Device Plugin 설정
- GPU 리소스 쿼터 및 스케줄링
- 학습 작업의 오토스케일링 설정
- 비용 최적화 (Spot/Preemptible 인스턴스 활용)

---

### 4.2 ML 파이프라인 CI/CD

```
코드 변경 ──> 데이터 검증 ──> 피처 생성 ──> 모델 학습
    │                                        │
    │                                        v
    │         모델 레지스트리 <── 모델 평가/검증
    │              │
    │              v
    └──> 스테이징 배포 ──> 카나리 배포 ──> 프로덕션 배포
```

**핵심 도구**:

| 도구 | 역할 | DevOps 대응 도구 |
|------|------|----------------|
| **Kubeflow Pipelines** | ML 워크플로우 오케스트레이션 | Jenkins Pipeline, Argo CD |
| **Apache Airflow** | 데이터/학습 파이프라인 스케줄링 | Cron, Jenkins |
| **MLflow** | 실험 추적, 모델 레지스트리 | Git + Docker Registry |
| **DVC** | 데이터/모델 버전 관리 | Git LFS |
| **Feast** | 피처 스토어 | Redis, 캐시 시스템 |

---

### 4.3 모델 서빙 인프라

| 서빙 도구 | 특징 | 사용 사례 |
|-----------|------|----------|
| **TensorFlow Serving** | TF 모델 특화, gRPC/REST | TensorFlow 기반 프로젝트 |
| **NVIDIA Triton** | 멀티 프레임워크, 고성능 | 대규모 추론 서비스 |
| **Seldon Core** | K8s 네이티브, A/B 테스트 내장 | K8s 환경에서의 ML 서빙 |
| **BentoML** | Python 친화적, 간편한 패키징 | 빠른 프로토타이핑 |
| **KServe** | K8s 표준, 서버리스 추론 | 클라우드 네이티브 환경 |

> **DevOps 엔지니어에게 좋은 소식**: 이미 알고 있는 K8s, Docker, CI/CD, 모니터링 지식이  
> MLOps에서 **그대로** 활용됩니다. ML 도메인 지식만 추가하면 됩니다.

## 5. 이 프로젝트를 실무에 적용한다면?

우리가 이번 교육에서 다룬 부동산 가격 예측 프로젝트를  
실제 프로덕션 서비스로 만든다면 어떻게 변해야 할까요?

### 전환 과정

```
    교육 (현재)                          실무 (목표)
    ──────────                          ──────────

    Jupyter Notebook          ──>       Python 스크립트 + 모듈화

    수동 실행                  ──>       자동화된 파이프라인
    (셀 하나씩 실행)                    (Airflow/Kubeflow)

    로컬 학습                  ──>       분산 학습
    (내 노트북 1대)                     (GPU 클러스터)

    CSV 파일 제출              ──>       REST API 서빙
                                        (실시간 가격 예측)

    일회성 결과                ──>       지속적 모니터링
                                        (성능 저하 알림)

    고정 데이터                ──>       실시간 데이터 수집
                                        (새로운 거래 데이터 반영)
```

### 구체적인 변환 예시

| 현재 (교육) | 실무 전환 | 담당 |
|------------|----------|------|
| `train.csv` 파일 로드 | DB에서 실시간 데이터 조회 | 데이터 엔지니어 |
| 노트북에서 피처 생성 | 피처 스토어 (Feast) 활용 | ML 엔지니어 + DevOps |
| `model.fit()` 로컬 실행 | Kubeflow Pipeline으로 분산 학습 | **DevOps 엔지니어** |
| `model.predict()` 로컬 실행 | TF Serving/Triton으로 API 서빙 | **DevOps 엔지니어** |
| CV 점수 확인 | MLflow로 실험 추적 + 모델 비교 | ML 엔지니어 |
| 없음 | Grafana 대시보드로 성능 모니터링 | **DevOps 엔지니어** |
| 없음 | 데이터 드리프트 감지 + 재학습 트리거 | ML 엔지니어 + **DevOps** |

---

# Part 2: 실습 & 토론 (50분)

## 실습 1: ML 파이프라인 전체 다이어그램

아래는 우리 프로젝트를 실무 ML 파이프라인으로 구성했을 때의 전체 흐름입니다.

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│                          ML 파이프라인 전체 흐름                                  │
│                                                                                 │
│                                                                                 │
│   ┌────────┐    ┌──────────────┐    ┌──────────┐    ┌──────────┐               │
│   │ Data   │───>│   Feature    │───>│Training  │───>│Evaluation│               │
│   │        │    │ Engineering  │    │          │    │          │               │
│   │ 데이터  │    │  피처        │    │  모델    │    │  모델    │               │
│   │ 수집   │    │  엔지니어링   │    │  학습    │    │  평가    │               │
│   └────────┘    └──────────────┘    └──────────┘    └─────┬────┘               │
│       ^                                                   │                     │
│       │                                                   v                     │
│       │              ┌──────────────────────────────────────┐                   │
│       │              │         Model Registry               │                   │
│       │              │         모델 레지스트리                │                   │
│       │              └──────────────┬───────────────────────┘                   │
│       │                             │                                           │
│       │                             v                                           │
│       │                     ┌──────────────┐    ┌──────────────┐               │
│       │                     │  Deployment  │───>│  Monitoring  │               │
│       │                     │              │    │              │               │
│       │                     │  모델 배포    │    │  모니터링     │               │
│       │                     └──────────────┘    └──────┬───────┘               │
│       │                                                │                       │
│       └────────────────────────────────────────────────┘                       │
│                          (재학습 트리거)                                         │
│                                                                                 │
└─────────────────────────────────────────────────────────────────────────────────┘
```

**각 단계별 DevOps 엔지니어의 역할**:

1. **Data (데이터 수집)**: 데이터 파이프라인 인프라 구축 (Kafka, Airflow)
2. **Feature Engineering (피처 엔지니어링)**: 피처 스토어 인프라 운영
3. **Training (모델 학습)**: GPU 클러스터 관리, 분산 학습 환경 제공
4. **Evaluation (모델 평가)**: CI 파이프라인에서 자동 평가 실행
5. **Model Registry (모델 레지스트리)**: MLflow 서버 운영
6. **Deployment (모델 배포)**: K8s 기반 서빙 인프라, 배포 자동화
7. **Monitoring (모니터링)**: Prometheus + Grafana 대시보드 구축

## 실습 2: 이 프로젝트의 MLOps 구성 예시

우리가 교육에서 만든 LightGBM 모델을 MLflow로 관리한다면 어떤 모습일까요?  
아래는 개념적인 예시 코드입니다.

In [ ]:
# MLflow 실험 추적 예시 (개념 코드 - 실행하지 않아도 됨)
# 실제로는 mlflow 패키지 설치와 서버 설정이 필요합니다.

"""
import mlflow
import lightgbm as lgb

# MLflow 실험 시작
mlflow.set_experiment("kakr-house-price")

with mlflow.start_run(run_name="lightgbm_v1"):
    # 1. 하이퍼파라미터 기록
    params = {
        "num_leaves": 15,
        "learning_rate": 0.05,
        "n_estimators": 300,
        "max_depth": -1,
        "subsample": 0.7,
    }
    mlflow.log_params(params)

    # 2. 모델 학습
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)

    # 3. 성능 메트릭 기록
    cv_score = 114912.07  # 4회차에서 구한 교차 검증 점수
    mlflow.log_metric("cv_rmse", cv_score)
    mlflow.log_metric("cv_mae", 85000.0)  # 예시 값

    # 4. 모델 아티팩트 저장 (모델 레지스트리에 등록)
    mlflow.sklearn.log_model(model, "lightgbm_model")

    # 5. 피처 중요도 차트 저장
    fig = lgb.plot_importance(model)
    mlflow.log_figure(fig, "feature_importance.png")

    print(f"Run ID: {mlflow.active_run().info.run_id}")
    print(f"CV RMSE: {cv_score:,.2f}")
"""

print("위 코드는 MLflow 실험 추적의 개념을 보여주는 예시입니다.")
print("실제 실행을 위해서는 MLflow 서버 설정이 필요합니다.")

In [ ]:
# 모델 서빙 API 예시 (개념 코드 - 실행하지 않아도 됨)
# FastAPI를 사용한 실시간 가격 예측 API

"""
from fastapi import FastAPI
import mlflow
import pandas as pd

app = FastAPI(title="부동산 가격 예측 API")

# MLflow에서 프로덕션 모델 로드
model = mlflow.sklearn.load_model("models:/kakr-house-price/Production")

@app.post("/predict")
async def predict_price(features: dict):
    \"\"\"
    부동산 가격 예측 엔드포인트
    
    입력 예시:
    {
        "size": 85.5,
        "floor": 12,
        "year_built": 2015,
        "latitude": 37.5,
        "longitude": 127.0
    }
    \"\"\"
    # 피처 변환
    df = pd.DataFrame([features])
    
    # 예측
    prediction = model.predict(df)[0]
    
    return {
        "predicted_price": int(prediction),
        "model_version": "v1.0",
        "confidence": "high"  # 실제로는 불확실성 추정 필요
    }
"""

print("위 코드는 FastAPI를 사용한 모델 서빙의 개념을 보여주는 예시입니다.")
print("실제 배포 시에는 Docker 컨테이너로 패키징하고 K8s에 배포합니다.")

In [ ]:
# Docker + K8s 배포 설정 예시 (개념 코드 - 실행하지 않아도 됨)

dockerfile_example = """
# === Dockerfile ===
FROM python:3.11-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt

COPY . .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]
"""

k8s_deployment_example = """
# === K8s Deployment (deployment.yaml) ===
apiVersion: apps/v1
kind: Deployment
metadata:
  name: house-price-predictor
spec:
  replicas: 3
  selector:
    matchLabels:
      app: house-price-predictor
  template:
    metadata:
      labels:
        app: house-price-predictor
    spec:
      containers:
      - name: predictor
        image: registry.example.com/house-price-predictor:v1.0
        ports:
        - containerPort: 8080
        resources:
          requests:
            memory: "512Mi"
            cpu: "500m"
          limits:
            memory: "1Gi"
            cpu: "1000m"
        readinessProbe:
          httpGet:
            path: /health
            port: 8080
          initialDelaySeconds: 10
          periodSeconds: 5
"""

print("=== Dockerfile 예시 ===")
print(dockerfile_example)
print("=== K8s Deployment 예시 ===")
print(k8s_deployment_example)
print("\n이미 익숙한 Docker/K8s 패턴을 ML 모델 서빙에 그대로 적용할 수 있습니다.")

## 실습 3: DevOps vs MLOps 비교표

DevOps에서 이미 알고 있는 개념들이 MLOps에서 어떻게 대응되는지 정리했습니다.

| DevOps 개념 | MLOps 대응 | 설명 |
|------------|-----------|------|
| CI/CD 파이프라인 | ML 파이프라인 | 코드 빌드/배포 대신 데이터 처리/모델 학습/배포 |
| 코드 버전 관리 (Git) | 데이터 + 모델 버전 관리 (DVC, MLflow) | 코드뿐 아니라 데이터와 모델도 버전 관리 |
| 모니터링 (CPU, 메모리, 응답시간) | 모델 성능 + 데이터 드리프트 모니터링 | 인프라 메트릭에 더해 모델 품질 메트릭 추가 |
| 로드 밸런싱 | 모델 A/B 테스트 | 트래픽 분배를 모델 비교 실험에 활용 |
| 인프라 스케일링 | GPU 클러스터 오토스케일링 | CPU 외에 GPU 리소스도 스케일링 대상 |
| 아티팩트 저장소 (Docker Registry) | 모델 레지스트리 (MLflow) | 도커 이미지 대신 학습된 모델을 저장/관리 |
| 설정 관리 (ConfigMap, Vault) | 하이퍼파라미터 관리 | 시스템 설정값 대신 모델 학습 설정값 관리 |
| 카나리 배포 | 모델 카나리 배포 | 새 모델을 일부 트래픽에만 적용하고 검증 |
| 롤백 | 모델 롤백 | 새 모델 성능이 나쁘면 이전 모델로 즉시 복귀 |
| IaC (Terraform) | ML 인프라 코드화 | ML 파이프라인도 코드로 정의하고 관리 |

## 실습 4: 전체 교육 회고

5회에 걸친 교육에서 우리가 함께 배운 내용을 정리합니다.

### 전체 교육 여정

```
1회차                2회차                3회차                4회차               5회차
ML 기본 + EDA        피처 엔지니어링       지리 데이터           모델 학습            MLOps
                                        피처 엔지니어링       교차 검증
  │                    │                    │                  │                   │
  v                    v                    v                  v                   v
┌──────┐           ┌──────┐           ┌──────┐           ┌──────┐           ┌──────┐
│데이터 │──────────>│피처   │──────────>│고급   │──────────>│모델   │──────────>│실무   │
│이해   │           │생성   │           │피처   │           │구축   │           │적용   │
└──────┘           └──────┘           └──────┘           └──────┘           └──────┘
```

### 각 회차별 핵심 내용

| 회차 | 주제 | 핵심 내용 | DevOps 연결점 |
|------|------|----------|---------------|
| **1회차** | ML 기본 개념 + EDA | 데이터 탐색, 분포 분석, 결측치 처리 | 로그 분석, 메트릭 탐색과 유사 |
| **2회차** | 피처 엔지니어링 | 원본 데이터에서 유의미한 변수 생성 | 파생 메트릭 생성과 유사 |
| **3회차** | 지리 데이터 피처 엔지니어링 | 위치 정보 활용, 공간 분석 | 인프라 위치 기반 최적화와 유사 |
| **4회차** | 모델 학습과 교차 검증 | LightGBM 학습, K-Fold CV, 하이퍼파라미터 | CI (빌드+테스트), 스테이징 검증과 유사 |
| **5회차** | MLOps와 실무 적용 | ML 파이프라인, 서빙, 모니터링 | DevOps와 직접적으로 연결 |

## 실습 5: DevOps 비유 총정리표

교육 전체에서 사용한 DevOps 비유를 한눈에 정리합니다.  
ML을 처음 접하는 DevOps 엔지니어에게 가장 유용한 참고 자료입니다.

| ML 개념 | DevOps 비유 | 설명 |
|---------|------------|------|
| **학습 데이터** | 과거 로그/메트릭 | 과거 데이터에서 패턴을 학습 |
| **테스트 데이터** | 프로덕션 트래픽 | 본 적 없는 데이터에 대한 성능 검증 |
| **피처 엔지니어링** | 파생 메트릭 생성 | 원본 데이터에서 유용한 정보를 추출/변환 |
| **모델 학습** | CI (빌드 + 테스트) | 데이터를 입력으로 모델(아티팩트)을 생성 |
| **교차 검증** | 스테이징 환경 테스트 | 여러 환경에서 반복 검증하여 안정성 확인 |
| **과적합** | 알람이 너무 민감한 상태 | 학습 데이터에만 과도하게 맞춰진 상태 |
| **과소적합** | 알람이 너무 둔감한 상태 | 패턴을 충분히 학습하지 못한 상태 |
| **하이퍼파라미터** | 시스템 설정값 | 모델 동작을 제어하는 외부 설정 |
| **하이퍼파라미터 튜닝** | 성능 튜닝 (JVM 옵션, 커널 파라미터) | 최적의 설정값을 찾는 과정 |
| **모델 배포** | CD (프로덕션 배포) | 학습된 모델을 서비스에 반영 |
| **데이터 드리프트** | 트래픽 패턴 변화 | 입력 데이터의 분포가 시간에 따라 변화 |
| **모델 성능 저하** | 서비스 성능 저하 (SLA 위반) | 예측 품질이 시간에 따라 떨어지는 현상 |
| **앙상블** | 로드 밸런서 뒤의 여러 서버 | 여러 모델의 예측을 결합하여 성능 향상 |
| **피처 스토어** | Redis/캐시 시스템 | 자주 사용하는 피처를 미리 계산하여 저장 |
| **모델 레지스트리** | Docker Registry / Artifactory | 학습된 모델의 버전 관리 및 저장소 |
| **재학습** | 핫픽스 배포 | 성능 저하 시 새 데이터로 모델을 다시 학습 |

---

# Part 3: 토론 주제

아래 주제들에 대해 자유롭게 토론해 봅시다.  
DevOps 경험을 바탕으로 ML 시스템에 대한 의견을 나눠보세요.

---

### 토론 1: "프로덕션 배포 시 어떤 인프라가 필요한가?"

우리의 부동산 가격 예측 모델을 실제 서비스로 배포한다고 가정합시다.

**생각해볼 질문들**:
- 어떤 클라우드 서비스를 사용할 것인가? (AWS, GCP, Azure)
- K8s 클러스터 구성은 어떻게 할 것인가?
- GPU가 필요한가? 필요하다면 학습용/추론용 각각 어떤 GPU를 선택할 것인가?
- 예상 트래픽에 따른 오토스케일링 설정은?
- 비용 최적화 방안은?

---

### 토론 2: "모델 성능 저하 시 어떤 메트릭을 모니터링해야 하는가?"

모델이 배포된 후 시간이 지나면서 성능이 저하될 수 있습니다.

**생각해볼 질문들**:
- 어떤 메트릭을 대시보드에 표시할 것인가?
- 알림(Alert)의 임계값은 어떻게 설정할 것인가?
- 성능 저하가 감지되면 자동으로 어떤 조치를 취할 것인가?
- 기존 APM 도구 (Datadog, New Relic 등)와 어떻게 통합할 것인가?
- 모델 성능 저하와 인프라 장애를 어떻게 구분할 것인가?

---

### 토론 3: "데이터 드리프트가 발생했을 때 대응 방법은?"

코로나 같은 외부 이벤트로 부동산 시장 패턴이 완전히 바뀌었다고 가정합시다.

**생각해볼 질문들**:
- 데이터 드리프트를 어떻게 감지할 것인가? (통계적 테스트? 규칙 기반?)
- 감지 후 자동 재학습을 트리거할 것인가, 아니면 수동 개입이 필요한가?
- 재학습 파이프라인의 SLA는 어떻게 설정할 것인가?
- 재학습 중 서비스 중단 없이 새 모델로 전환하는 방법은?
- 롤백 전략은 어떻게 구성할 것인가?

---

# 추가 학습 자료

MLOps에 대해 더 깊이 공부하고 싶다면 아래 자료들을 참고하세요.

### 입문 자료

- **Google ML Engineering Best Practices**: Google에서 제공하는 ML 엔지니어링 가이드
- **MLOps: Continuous delivery and automation pipelines in machine learning**: Google Cloud 공식 MLOps 문서
- **Hidden Technical Debt in Machine Learning Systems** (논문): ML 시스템의 기술 부채에 관한 Google 논문

### 도구별 공식 문서

- **MLflow 공식 문서** (mlflow.org): 실험 추적, 모델 레지스트리
- **Kubeflow 공식 문서** (kubeflow.org): K8s 기반 ML 파이프라인
- **Airflow 공식 문서** (airflow.apache.org): 워크플로우 오케스트레이션
- **DVC 공식 문서** (dvc.org): 데이터/모델 버전 관리
- **Feast 공식 문서** (feast.dev): 피처 스토어

### 실습 프로젝트

- **Made With ML**: MLOps 전체 파이프라인을 실습할 수 있는 오픈소스 프로젝트
- **MLOps Zoomcamp**: DataTalks.Club에서 제공하는 무료 MLOps 교육 과정
- **Full Stack Deep Learning**: 실무 ML 시스템 구축을 다루는 교육 과정

### 커뮤니티

- **MLOps Community** (mlops.community): MLOps 실무자 커뮤니티
- **Awesome MLOps** (GitHub): MLOps 관련 도구와 자료 모음

---

# 다음 단계

5회 교육을 마무리하며, 앞으로의 학습 방향을 제안합니다.

### 단기 (1-2주)

1. **MLflow 로컬 설치 및 실습**
   - 이번 교육에서 만든 모델을 MLflow로 실험 추적해보기
   - `pip install mlflow` 후 `mlflow ui`로 대시보드 확인

2. **노트북을 Python 스크립트로 변환**
   - 데이터 로드, 피처 엔지니어링, 모델 학습을 각각 모듈로 분리
   - argparse로 하이퍼파라미터를 커맨드라인에서 받을 수 있게 변경

### 중기 (1-2개월)

3. **Docker로 모델 서빙 패키징**
   - FastAPI + 학습된 모델을 Docker 이미지로 패키징
   - 로컬에서 API 테스트

4. **간단한 ML 파이프라인 구축**
   - Airflow 또는 Prefect로 데이터 처리 → 학습 → 평가 파이프라인 자동화
   - GitHub Actions로 CI 파이프라인에 모델 테스트 추가

### 장기 (3-6개월)

5. **K8s 기반 ML 인프라 구축**
   - Kubeflow 또는 KServe로 모델 서빙 인프라 구축
   - GPU 노드 관리 경험

6. **모니터링 대시보드 구축**
   - Prometheus + Grafana로 모델 성능 모니터링
   - Evidently AI로 데이터 드리프트 감지

7. **Kaggle 대회 참여**
   - 새로운 대회에 참여하여 ML 스킬 향상
   - 대회 결과물을 MLOps 파이프라인으로 구축하는 것까지 연습

---

### 마무리

5회에 걸친 교육을 통해 우리는 다음을 달성했습니다:

- ML의 기본 개념을 **DevOps 언어로** 이해
- 실제 데이터로 **EDA부터 모델 학습까지** 전 과정 실습
- ML과 DevOps의 연결점인 **MLOps**를 파악
- DevOps 엔지니어로서 MLOps에서 **어떤 역할을 할 수 있는지** 이해

> **기억하세요**: ML 시스템의 95%는 인프라입니다.  
> 여러분은 이미 그 95%를 다루는 전문가입니다.  
> ML의 5%를 이해한 지금, 여러분은 MLOps의 핵심 인력이 될 준비가 되었습니다.

수고하셨습니다!